# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to explore the FAIR² dataset of clinicopathological and molecular characteristics of second primary colorectal cancer in cancer survivors, utilizing the [`mlcroissant`](https://github.com/mlcommons/croissant) library for Python. We will walk through metadata loading, exploration of record sets and fields (always referenced by `@id`), and perform basic exploratory data analysis and visualization.

### Dataset Source
The Croissant schema for the dataset is provided at the following URL:

[https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure mlcroissant and dependencies are installed
!pip install mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load dataset and metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All references to record sets, fields, and columns use their `@id`.

In [ ]:
# List available record sets using their @id
record_sets = list(dataset.record_sets)
print("Available Record Sets:")
for rset in record_sets:
    print(f"  - @id: {rset['@id']}  |  name: {rset.get('name', '--')}")

# For further exploration, select the primary record set by @id.
if len(record_sets) == 0:
    raise Exception("No record sets found in this dataset.")
# For this dataset, let's use the first found record set (replace with actual @id if known)
record_set_id = record_sets[0]['@id']

print(f"\nFields for record set @id={record_set_id}:")
fields = record_sets[0].get('field', [])
if isinstance(fields, dict):
    fields = [fields]
for f in fields:
    if isinstance(f, dict):
        print(f"  - Field @id: {f.get('@id', '--')}  |  name: {f.get('name', '--')}")
    else:
        print(f"  - Field: {f}")


Below is an example of records from the chosen record set using its `@id`. Explore individual entries and review their field names (by `@id`).

In [ ]:
# Preview a few records (as dicts) from the selected record set
print(f"\nExample records from record set @id={record_set_id}:")
for i, rec in enumerate(dataset.records(record_set=record_set_id)):
    print(rec)
    if i >= 2:
        break

## 3. Data Extraction
Load data from each record set into a DataFrame. Use the record set and field `@id`s from above.

In [ ]:
# List @id of all record sets
record_set_ids = [rset['@id'] for rset in record_sets]

dataframes = {}
for rsid in record_set_ids:
    print(f"Loading records for record set: {rsid}")
    df = pd.DataFrame(list(dataset.records(record_set=rsid)))
    print(f" - shape: {df.shape}\n")
    dataframes[rsid] = df

# For further analysis, select the main clinical tabular record set
main_rs_id = record_set_ids[0]
main_df = dataframes[main_rs_id]
print(f"Columns in main record set (@id={main_rs_id}):\n{list(main_df.columns)}")
main_df.head()

## 4. Exploratory Data Analysis (EDA)
We now explore the dataset. We'll select a numeric field (by `@id`) for examples of filtering, normalization, and grouping, referencing entities only by their `@id` throughout.

**Tip:** If you want to see all field/column @id's, inspect `main_df.columns` above.

In [ ]:
# Pick a numeric field for demonstration (ensure this @id matches your dataset field)
# For illustration, let's suppose there's a numeric field '@id': 'age_at_diagnosis'
numeric_field_id = 'age_at_diagnosis'  # Use the correct @id for age at diagnosis from main_df.columns

if numeric_field_id not in main_df.columns:
    print(f"Field '{numeric_field_id}' not found in columns. Available columns are: {main_df.columns.tolist()}")
else:
    threshold = 40
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalizing field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Grouping example - use a categorical field (e.g., 'sex', referenced by its @id)
    group_field_id = 'sex'  # Use correct @id as found in columns
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
    else:
        print(f"Field '{group_field_id}' not found in columns for grouping.")

## 5. Visualization
Visualize distributions or relationships between selected fields. All entities referenced by their `@id`.

In [ ]:
# Example: Visualize the distribution of age at diagnosis, by sex
if numeric_field_id in main_df.columns:
    plt.figure(figsize=(8, 5))
    if 'sex' in main_df.columns:
        sns.histplot(data=main_df, x=numeric_field_id, kde=True, hue='sex', bins=15, element="step")
        plt.title('Distribution of Age at Diagnosis by Sex (@id: age_at_diagnosis, sex)')
        plt.xlabel('Age at Diagnosis')
        plt.ylabel('Count')
        plt.legend(title='Sex (@id: sex)')
    else:
        sns.histplot(main_df[numeric_field_id], bins=15, kde=True)
        plt.title('Distribution of Age at Diagnosis (@id: age_at_diagnosis)')
        plt.xlabel('Age at Diagnosis')
        plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print(f"Numeric field '{numeric_field_id}' not found in dataset. Cannot plot.")

## 6. Conclusion

This notebook demonstrated end-to-end exploration of the FAIR² dataset using only entity references by `@id` as required for robust, reproducible data-scientific workflow. We loaded dataset metadata, inspected record sets and fields, extracted tabular data, performed basic exploratory analysis (filtering, normalization, grouping), and visualized distributions. For further, more advanced analyses, refer to the official [mlcroissant documentation](https://mlcroissant.readthedocs.io/en/latest/).
